# 04_08 – Исключение 147 неоднозначных пар

По решению пользователя от 08.10.2026 исключаем обе строки всех 147 разобранных пар C40: 294 записи, включая три GoA=True. Это ограничение исследовательской выборки ради завершения разбора неоднозначных связей, а не доказательство ошибочности каждой строки.

Создаем отдельную копию 04_c40_ambiguous_pairs_excluded.parquet. Все прочие строки, значения и поля сохраняем. Исходник и копия 03 остаются неизменными; исключенные записи уже сохранены для аудита. Пропуски C40, поздние отметки вне этих пар и специальные полеты этим правилом не исключаем.

## Основание исключения

В 147 парах две строки одного самолета и аэропорта имеют одинаковые время и координаты C40, но разные времена завершения и идентификаторы наблюдений. Без дополнительного восстановления траекторий однозначная связь общей точки прогноза с отдельным наблюдением и его целевой меткой не подтверждена. Такие строки могут смешивать границы событий и повторно учитывать одну точку прогноза.

Проверка одной пары по публичным измерениям выявила несоответствие раннего landing_time нахождению самолета в воздухе. Это подтверждает необходимость осторожности, но не доказывает ошибочность остальных 146 пар. Материалы проверки: [04_05](../../reports/analytical_sample/04_05_pair_review.csv) и [04_07](../../reports/analytical_sample/04_07_review.md).

Поэтому применяем консервативное исследовательское правило: исключаем обе строки каждой неоднозначной пары, не выбирая предположительно правильную строку и не исправляя метки без достаточных свидетельств. Решение принято с учетом затрат на дальнейшую проверку. Это ограничение области эксперимента, а не авиационная норма и не доказательство ошибочности всех исключенных записей.

Исключено 294 из 7 984 254 строк (0,00368%) и 3 из 26 253 положительных меток (0,01143%). Малый объем исключения не гарантирует отсутствия смещения; выводы эксперимента относятся к выборке после этого правила. Исключенные записи сохранены для возможного последующего пересмотра.


In [1]:
# Находим проект и подключаем согласованное преобразование
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents]
          if (p/'europe/data/source/osn23_landings_merged.parquet.gzip').exists())
sys.path.insert(0,str(ROOT/'europe/scripts'))
import exclude_ambiguous_pairs as preparation

# Создаем копию и проверяем каждую сохраненную строку относительно входа
# Повторный запуск при существующем выходе остановится, не перезаписывая файл
result=preparation.main()
display(pd.DataFrame([result]))

{
  "rule": "exclude both rows of all 147 reviewed ambiguous C40 pairs",
  "decision_date": "2026-10-08",
  "authorized_by": "user",
  "input": "europe\\data\\working\\03_c40_diagnostic_flags.parquet",
  "input_sha256": "ddf11438bab44137242ffcf4da928418e116efd46defce58dc8a388b67942b6d",
  "input_unchanged": true,
  "output": "europe\\data\\working\\04_c40_ambiguous_pairs_excluded.parquet",
  "output_sha256": "ba5c3c4686f76a57384276d2c27f0fdcdc7b6d58657fdfb7c7cc8ab8c7b99e05",
  "rows_before": 7984254,
  "rows_after": 7983960,
  "rows_excluded": 294,
  "go_arounds_before": 26253,
  "go_arounds_after": 26250,
  "go_arounds_excluded": 3,
  "physical_columns": 31,
  "values_corrected": 0,
  "preserved_rows_exact_comparison_passed": true,
  "excluded_records_archive": "europe/data/review/04_05_c40_pair_full_records.parquet",
  "reason": "research exclusion of unresolved pairs; does not establish all rows as incorrect",
  "missing_c40_filter_applied": false,
  "all_other_flights_retained": tr

,rule,decision_date,authorized_by,input,input_sha256,input_unchanged,output,output_sha256,rows_before,rows_after,...,go_arounds_before,go_arounds_after,go_arounds_excluded,physical_columns,values_corrected,preserved_rows_exact_comparison_passed,excluded_records_archive,reason,missing_c40_filter_applied,all_other_flights_retained
0,exclude both rows of all 147 reviewed ambiguou...,2026-10-08,user,europe\data\working\03_c40_diagnostic_flags.pa...,ddf11438bab44137242ffcf4da928418e116efd46defce...,True,europe\data\working\04_c40_ambiguous_pairs_exc...,ba5c3c4686f76a57384276d2c27f0fdcdc7b6d58657fdf...,7984254,7983960,...,26253,26250,3,31,0,True,europe/data/review/04_05_c40_pair_full_records...,research exclusion of unresolved pairs; does n...,False,True


## Результат и дальнейшая работа

Остается 7 983 960 строк и 26 250 меток GoA=True. Состав полей и значения сохраненных записей не меняются. Это актуальная рабочая копия дальнейших шагов, но не окончательный набор признаков и не полностью проверенный C40-эксперимент.

Разбор этих 147 пар закрыт; дополнительных загрузок для них не требуется. Далее анализируем оставшиеся проблемы и согласуем самостоятельные правила очистки. [Журнал решения и проверки](../../reports/analytical_sample/04_08_pairs_excluded.json).